# 🤖 Ollama on Google Colab (GPU T4)
**Model:** `yxchia/seallms-v3-7b:Q4_K_M`  
**Purpose:** Run Ollama on Colab GPU → expose via ngrok → connect Go RAG app

### Steps:
1. Install Ollama
2. Pull the SEA LLM model
3. Expose via ngrok (free public URL)
4. Copy the URL → paste into Go app

## ✅ Step 1 — Check GPU

In [ ]:
!nvidia-smi

## ✅ Step 2 — Install Ollama

In [ ]:
# Install zstd first (required by new Ollama versions)
!apt-get install -y zstd

# Then install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

## ✅ Step 3 — Start Ollama server (background)

In [ ]:
import subprocess
import time

# Start Ollama server in the background
# OLLAMA_HOST=0.0.0.0 so ngrok can reach it
proc = subprocess.Popen(
    ['ollama', 'serve'],
    env={**__import__('os').environ, 'OLLAMA_HOST': '0.0.0.0:11434'},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)
print('✅ Ollama server started (PID:', proc.pid, ')')

## ✅ Step 4 — Pull the SEA LLM model
> ⚠️ 7B Q4_K_M model = ~4.1 GB download. Takes ~3-5 minutes on Colab.

In [ ]:
!ollama pull yxchia/seallms-v3-7b:Q4_K_M

## ✅ Step 5 — Pull the Embedding model

In [ ]:
!ollama pull nomic-embed-text

## ✅ Step 6 — Test Ollama is working

In [ ]:
import requests

res = requests.post('http://localhost:11434/api/chat', json={
    'model': 'yxchia/seallms-v3-7b:Q4_K_M',
    'messages': [{'role': 'user', 'content': 'Hello! Who are you?'}],
    'stream': False
})
print(res.json()['message']['content'])

## ✅ Step 7 — Expose with ngrok
> **Get free token:** https://dashboard.ngrok.com/signup  
> Sign up → **Your Authtoken** → copy & paste below

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok

# ⬇️ Paste your ngrok authtoken here
NGROK_TOKEN = "3ACOWs1OOGjQbbWPwDdtd8od9HC_4Hz3j5VwiEvRPPfEETLUC"

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(11434)

ollama_url = tunnel.public_url
print('='*60)
print('✅ Ollama is now public!')
print()
print('OLLAMA_URL =', ollama_url)
print()
print('Run your Go app with:')
print(f'  OLLAMA_URL={ollama_url} go run .')
print('='*60)

## ✅ Step 8 — Keep Colab alive (run this last)
> Colab disconnects after idle. This cell keeps it busy.

In [ ]:
import time

print('🔄 Keeping Colab alive... Press Stop to end.')
while True:
    time.sleep(60)
    print('.', end='', flush=True)